In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

from tensorflow.keras import layers, models, optimizers, utils
from tensorflow.keras.preprocessing import image_dataset_from_directory

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.19.0
TensorFlow Version: 2.19.0


In [8]:
data_dir = "/content/drive/MyDrive/Bone Break Classification/Bone Break Classification"

train_data = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(256, 256),
    batch_size=32
)

validation_data = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(256, 256),
    batch_size=32
)

class_names = train_data.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

Found 1129 files belonging to 10 classes.
Using 904 files for training.
Found 1129 files belonging to 10 classes.
Using 225 files for validation.
Classes: ['Avulsion fracture', 'Comminuted fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']


In [9]:

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_data = train_data.map(preprocess)
validation_data = validation_data.map(preprocess)

def dataset_to_numpy(dataset):
    x, y = [], []
    for images, labels in dataset:
        x.append(images.numpy())
        y.append(labels.numpy())
    return np.concatenate(x), np.concatenate(y)

x_train, y_train = dataset_to_numpy(train_data)
x_val, y_val = dataset_to_numpy(validation_data)

y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_val = utils.to_categorical(y_val, NUM_CLASSES)

In [10]:
IMAGE_SIZE = 256
PATCH_SIZE = 16
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 6

In [11]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1,1,1,1],
            padding="VALID",
        )
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dim])
        return patches

In [12]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(num_patches, projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=NUM_PATCHES, delta=1)
        encoded = self.projection(patches) + self.position_embedding(positions)
        return encoded

In [13]:
inputs = layers.Input(shape=(256,256,3))

patches = Patches(PATCH_SIZE)(inputs)
encoded = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)

for _ in range(TRANSFORMER_LAYERS):

    # Layer Normalization + Attention
    x1 = layers.LayerNormalization(epsilon=1e-6)(encoded)
    attention = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=PROJECTION_DIM
    )(x1, x1)

    x2 = layers.Add()([attention, encoded])

    # MLP Block
    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    x3 = layers.Dense(PROJECTION_DIM*2, activation="relu")(x3)
    x3 = layers.Dense(PROJECTION_DIM)(x3)

    encoded = layers.Add()([x3, x2])

# Classification Head
representation = layers.LayerNormalization(epsilon=1e-6)(encoded)
representation = layers.GlobalAveragePooling1D()(representation)
representation = layers.Dense(256, activation="relu")(representation)
representation = layers.Dropout(0.5)(representation)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(representation)

model_vit = models.Model(inputs, outputs)
model_vit.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patches (Patches)   │ (None, None, 768) │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder       │ (None, 256, 64)   │     65,600 │ patches[0][0]     │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 256, 64)   │        128 │ patch_encoder[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 256, 64)   │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256, 64)   │          0 │ multi_head_atten… │
│                     │                   │            │ patch_encoder[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256, 64)   │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256, 64)   │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256, 64)   │          0 │ dense_4[0][0],    │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256, 64)   │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 256, 64)   │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 256, 64)   │          0 │ multi_head_atten… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256, 64)   │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256, 64)   │      8,256 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 256, 64)   │          0 │ dense_6[0][0],    │
│                     │                   │            │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256, 64)   │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 584,138 (2.23 MB)

 Trainable params: 584,138 (2.23 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
optimizer = optimizers.RMSprop(learning_rate=0.0001)

model_vit.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model_vit.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=100,
    validation_data=(x_val, y_val),
    shuffle=True
)

Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 32s 501ms/step - accuracy: 0.1121 - loss: 2.4815 - val_accuracy: 0.1556 - val_loss: 2.2679
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.1346 - loss: 2.3225 - val_accuracy: 0.1244 - val_loss: 2.2833
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.1233 - loss: 2.2895 - val_accuracy: 0.1289 - val_loss: 2.2881
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.1098 - loss: 2.3145 - val_accuracy: 0.1289 - val_loss: 2.2517
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.1359 - loss: 2.2869 - val_accuracy: 0.1467 - val_loss: 2.2445
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.1365 - loss: 2.3045 - val_accuracy: 0.1689 - val_loss: 2.2423
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.1307 - loss: 2.3044 - val_accuracy: 0.1244 - val_loss: 2.2642
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.1325 - loss: 2.2862 - val_accuracy: 

In [15]:
y_pred_probs = model_vit.predict(x_val)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_val, axis=1)

acc   = accuracy_score(y_true_classes, y_pred_classes)
prec  = precision_score(y_true_classes, y_pred_classes, average='weighted')
rec   = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1    = f1_score(y_true_classes, y_pred_classes, average='weighted')
bal_acc = balanced_accuracy_score(y_true_classes, y_pred_classes)

print(f"\nAccuracy          : {acc:.4f}")
print(f"Precision         : {prec:.4f}")
print(f"Recall            : {rec:.4f}")
print(f"F1 Score          : {f1:.4f}")
print(f"Balanced Accuracy : {bal_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 254ms/step

Accuracy          : 0.1956
Precision         : 0.2547
Recall            : 0.1956
F1 Score          : 0.1793
Balanced Accuracy : 0.1801

Classification Report:
                       precision    recall  f1-score   support

    Avulsion fracture       0.20      0.30      0.24        20
  Comminuted fracture       0.33      0.17      0.23        29
 Fracture Dislocation       0.18      0.48      0.26        31
  Greenstick fracture       0.12      0.25      0.16        20
    Hairline Fracture       0.50      0.07      0.13        27
    Impacted fracture       0.00      0.00      0.00        18
Longitudinal fracture       0.18      0.11      0.14        18
     Oblique fracture       0.60      0.18      0.27        17
Pathological fracture       0.25      0.17      0.20        30
      Spiral Fracture       0.09      0.07      0.08        15

             accuracy                           0.20       225
            macro avg       0.24      0.18 